# Import Modules

In [ ]:
from utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.family'] = 'Tahoma'
plt.rcParams['axes.unicode_minus'] = False

# Validation

In [ ]:
issues_df = run_validation(load_ocr_data)

In [ ]:
issues_df = run_validation(load_ocr_clean_data)

# Clean Data

In [ ]:
dfs_district = []
dfs_partylist = []
dfs_referendum = []

for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if "district" in file_config["path"]: 
        for page_num in range(1, total_pages, 2):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_district.append(df_dict)
            
    elif "partylist" in file_config["path"]:
        for page_num in range(1, total_pages, 4):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_partylist.append(df_dict)
            
    elif "referendum" in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_referendum.append(df_dict)

In [ ]:
print(f"Total District Units: {len(dfs_district)}")
print(f"Total Partylist Units: {len(dfs_partylist)}")
print(f"Total Referendum Units: {len(dfs_referendum)}")

In [ ]:
district_summary, district_scores = flatten_district(
    dfs_district
)
partylist_summary, partylist_scores = flatten_partylist(
    dfs_partylist
)
referendum_summary, referendum_scores = flatten_referendum(
    dfs_referendum
)

district_scores["คะแนน"] = pd.to_numeric(district_scores["คะแนน"], errors="coerce").fillna(0).astype(int)
partylist_scores["คะแนน"] = pd.to_numeric(partylist_scores["คะแนน"], errors="coerce").fillna(0).astype(int)
referendum_scores["คะแนน"] = pd.to_numeric(referendum_scores["คะแนน"], errors="coerce").fillna(0).astype(int)

In [ ]:
district_summary.tail(5)

In [ ]:
district_scores.tail(10)

In [ ]:
partylist_summary.tail(5)

In [ ]:
partylist_scores.tail(10)

In [ ]:
referendum_summary.tail(5)

In [ ]:
referendum_scores.tail(5)

In [ ]:
district_summary.to_csv("flatten_result/district_summary.csv")
district_scores.to_csv("flatten_result/district_scores.csv")
partylist_summary.to_csv("flatten_result/partylist_summary.csv")
partylist_scores.to_csv("flatten_result/partylist_scores.csv")
referendum_summary.to_csv("flatten_result/referendum_summary.csv")
referendum_scores.to_csv("flatten_result/referendum_scores.csv")

# Insights

## สส.เขต

### 1. คะแนนรวมแต่ละผู้สมัคร

In [ ]:
d1 = (district_scores.groupby(["ชื่อผู้สมัคร", "พรรค"], as_index=False)["คะแนน"]
        .sum().sort_values("คะแนน", ascending=False).reset_index(drop=True))
d1.index += 1
d1["% คะแนน"] = (d1["คะแนน"] / d1["คะแนน"].sum() * 100).round(2)

display(d1)

fig, ax = plt.subplots(figsize=(10, max(4, len(d1) * 0.5)))
colors = ["red" if i == 0 else "blue" for i in range(len(d1))]
ax.barh(d1["ชื่อผู้สมัคร"] + " (" + d1["พรรค"] + ")", d1["คะแนน"], color=colors)
ax.set_xlabel("คะแนน")
ax.set_title("คะแนนรวมทุกหน่วย")
ax.invert_yaxis()
for i, v in enumerate(d1["คะแนน"]):
    ax.text(v + 10, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

### 2. จำนวนหน่วยที่ผู้สมัครแต่ละคนชนะ

In [ ]:
unit_winner = district_scores.loc[
    district_scores.groupby("unit_key")["คะแนน"].idxmax(), ["unit_key", "ชื่อผู้สมัคร", "พรรค"]
]
d2 = (unit_winner.groupby(["ชื่อผู้สมัคร", "พรรค"])
      .size().reset_index(name="หน่วยที่ชนะ")
      .sort_values("หน่วยที่ชนะ", ascending=False).reset_index(drop=True))
d2.index += 1
d2["% หน่วยที่ชนะ"] = (d2["หน่วยที่ชนะ"] / d2["หน่วยที่ชนะ"].sum() * 100).round(2)

display(d2)

fig, ax = plt.subplots(figsize=(10, max(4, len(d2) * 0.5)))
colors = ["red" if i == 0 else "blue" for i in range(len(d2))]
ax.barh(d2["ชื่อผู้สมัคร"] + " (" + d2["พรรค"] + ")", d2["หน่วยที่ชนะ"], color=colors)
ax.set_xlabel("จำนวนหน่วย")
ax.set_title("จำนวนหน่วยที่ชนะ")
ax.invert_yaxis()
for i, v in enumerate(d2["หน่วยที่ชนะ"]):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.show()

### 3. สัดส่วนการใช้สิทธิเลือกตั้งสส.เขตรายตำบล

In [ ]:
turnout_by_tambon = (
    district_summary.groupby("ตำบล").apply(lambda x: pd.Series({
        "จำนวนหน่วยเลือกตั้ง": x["จำนวนบัตรทั้งหมด"].count(),
        "บัตรทั้งหมด": x["จำนวนบัตรทั้งหมด"].sum(),
        "บัตรที่ถูกใช้": (x["บัตรดี"] + x["บัตรเสีย"] + x["ไม่เลือกผู้ใด"]).sum(),
    }))
).reset_index()
turnout_by_tambon["% บัตรที่ถูกใช้"] = (turnout_by_tambon["บัตรที่ถูกใช้"] / turnout_by_tambon["บัตรทั้งหมด"] * 100).round(2)
turnout_by_tambon["บัตรที่เหลือ"] = turnout_by_tambon["บัตรทั้งหมด"] - turnout_by_tambon["บัตรที่ถูกใช้"]
turnout_by_tambon = turnout_by_tambon.sort_values("% บัตรที่ถูกใช้", ascending=False)

display(turnout_by_tambon)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(turnout_by_tambon["ตำบล"],
       turnout_by_tambon["บัตรที่ถูกใช้"],
       color="green",
       label="จำนวนบัตรที่ถูกใช้")
ax.bar(turnout_by_tambon["ตำบล"],
       turnout_by_tambon["บัตรที่เหลือ"],
       bottom=turnout_by_tambon["บัตรที่ถูกใช้"],
       color="red",
       label="จำนวนบัตรที่เหลือ")
ax.set_xlabel("ตำบล")
ax.set_ylabel("จำนวน (บัตร)")
ax.set_title("สัดส่วนการใช้สิทธิเลือกตั้งรายตำบล")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 4. การกระจายคะแนนแต่ละพรรครายตำบล

In [ ]:
d4 = district_scores.groupby(["ตำบล", "พรรค"], as_index=False)["คะแนน"].sum()
d4_pivot = d4.pivot_table(index="ตำบล", columns="พรรค", values="คะแนน", fill_value=0)
d4_pivot.plot(kind="bar", figsize=(14, 6), title="การกระจายคะแนนสส.เขตรายตำบล (แยกพรรค)")

display(d4_pivot)

plt.ylabel("คะแนน")
plt.xlabel("ตำบล")
plt.xticks(rotation=45)
plt.legend(title="พรรค")
plt.tight_layout()
plt.show()

In [ ]:
pivot_const = district_scores.pivot_table(index="ตำบล", columns="พรรค", values="คะแนน",
                                          aggfunc="sum", fill_value=0)

pivot_const_pct = pivot_const.div(pivot_const.sum(axis=1), axis=0)

top_const = pivot_const_pct.sum().index
pivot_const_pct = pivot_const_pct[top_const]

plt.figure(figsize=(12,6))
sns.heatmap(pivot_const_pct, cmap="Blues")
plt.title("Heatmap พรรค × ตำบล (%)")
plt.show()

### 5. บัตรเสียรายตำบล

In [ ]:
tambon_bad = (district_summary.groupby(["ตำบล"])
              .agg({"จำนวนบัตรทั้งหมด": "sum", "บัตรเสีย": "sum", "unit_key": "count"}).reset_index())

tambon_bad.rename(columns={"unit_key": "จำนวนหน่วย"}, inplace=True)

tambon_bad["% บัตรเสีย"] = (tambon_bad["บัตรเสีย"] / tambon_bad["จำนวนบัตรทั้งหมด"] * 100).replace([np.inf, -np.inf], np.nan)

total_bad = tambon_bad["บัตรเสีย"].sum()
tambon_bad = tambon_bad.sort_values("บัตรเสีย", ascending=False)
display(tambon_bad)

### 6. พรรคที่ชนะในแต่ละตำบล

In [ ]:
total_votes = district_scores.groupby('ตำบล')['คะแนน'].sum().reset_index(name='total_votes')
party_votes = district_scores.groupby(['ตำบล', 'พรรค'])['คะแนน'].sum().reset_index()

merged = party_votes.merge(total_votes, on='ตำบล')
merged['สัดส่วนคะแนน'] = merged['คะแนน'] / merged['total_votes']

top_party = merged.sort_values('สัดส่วนคะแนน', ascending=False).drop_duplicates('ตำบล')

party_colors = {p: c for p, c in zip(district_scores['พรรค'].unique(),plt.cm.tab10.colors)}

colors = top_party['พรรค'].map(party_colors)

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(top_party)), top_party['สัดส่วนคะแนน'], color=colors)

for i, v in enumerate(top_party['สัดส่วนคะแนน']):
    plt.text(i, v + 0.005, f"{v*100:.1f}%", ha='center', va='bottom', fontsize=8)

plt.xticks(range(len(top_party)), top_party['ตำบล'], rotation=45)
plt.title('พรรคที่ได้คะแนนสูงสุดในแต่ละตำบล')
plt.xlabel('ตำบล')
plt.ylabel('สัดส่วนคะแนน')

handles = [plt.Rectangle((0,0),1,1, color=c) for c in party_colors.values()]
plt.legend(handles, party_colors.keys(), title='พรรค')

plt.tight_layout()
plt.show()

### 7. 5 หน่วยฐานเสียงแข็ง และ 5 หน่วยที่ต้องพัฒนาของแต่ละพรรค

In [ ]:
TOP_UNIT = 5

def calc_margin(group):
    group = group.sort_values("คะแนน", ascending=False).reset_index(drop=True)
    if len(group) < 2:
        return pd.Series({"winner": group.loc[0, "ชื่อผู้สมัคร"], "พรรค": group.loc[0, "พรรค"],
                          "winner_score": group.loc[0, "คะแนน"], "margin": group.loc[0, "คะแนน"], "margin_pct": 1.0})
    margin = group.loc[0, "คะแนน"] - group.loc[1, "คะแนน"]
    total = group["คะแนน"].sum()
    return pd.Series({"winner": group.loc[0, "ชื่อผู้สมัคร"], "พรรค": group.loc[0, "พรรค"],
                      "winner_score": group.loc[0, "คะแนน"], "margin": margin, "margin_pct": margin / total})

unit_margin = district_scores.groupby("unit_key").apply(calc_margin, include_groups=False).reset_index()
unit_margin = unit_margin.merge(district_scores[["unit_key", "หน่วย", "ตำบล", "อำเภอ"]].drop_duplicates(), on="unit_key")

parties = district_scores["พรรค"].unique()

for party in parties:
    own = unit_margin[unit_margin["พรรค"] == party].copy()
    lost = unit_margin[unit_margin["พรรค"] != party].copy()
    own_sorted = own.sort_values("margin_pct", ascending=False).head(TOP_UNIT).reset_index(drop=True)
    lost_sorted = lost.sort_values("margin_pct", ascending=False).head(TOP_UNIT).reset_index(drop=True)

    if own_sorted.empty and lost_sorted.empty:
        continue

    n_rows = max(len(own_sorted), len(lost_sorted), 1)
    fig_height = max(4, n_rows * 1.2)

    fig, axes = plt.subplots(1, 2, figsize=(12, fig_height))
    fig.suptitle(f"พรรค{party}", fontweight="bold")

    if not own_sorted.empty:
        own_sorted = own_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        print(own_sorted["margin_pct"].sort_values())
        values = own_sorted["margin_pct"] * 100
        axes[0].barh(own_sorted["ตำบล"] + " หน่วย " + own_sorted["หน่วย"], values, color="blue")
        axes[0].set_xlim(0, values.max() * 1.2)

    axes[0].set_xlabel("Margin (%)")
    axes[0].set_title(f"Top {TOP_UNIT} ฐานเสียงแข็ง")

    if not lost_sorted.empty:
        lost_sorted = lost_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        print(lost_sorted["margin_pct"].sort_values())
        values = lost_sorted["margin_pct"] * 100
        axes[1].barh(lost_sorted["ตำบล"] + " หน่วย " + lost_sorted["หน่วย"],values, color="red")
        axes[1].set_xlim(1, values.max() * 1.2)

    axes[1].set_xlabel("Margin (%)")
    axes[1].set_title(f"Top {TOP_UNIT} พื้นที่ที่แพ้ห่างที่สุด")

    plt.tight_layout()
    plt.show()

## บัญชีรายชื่อ

In [ ]:
TOP = 10

### 1. คะแนนรวมแต่ละพรรค (Top 10)

In [ ]:
party_total_top = partylist_scores.groupby("พรรค")["คะแนน"].sum().sort_values(ascending=False).head(TOP)
party_total_top_df = pd.DataFrame(party_total_top, columns=["คะแนน"])
party_total_top_df["% คะแนน"] = party_total_top_df["คะแนน"] / party_total_top_df["คะแนน"].sum() * 100
display(party_total_top_df)

plt.figure(figsize=(10,6))
party_total_top.plot(kind="bar")
plt.title(f"คะแนนรวมแต่ละพรรค (Top {TOP})")
plt.ylabel("คะแนน")
plt.xticks(rotation=45)
plt.show()

### 2. จำนวนหน่วยที่แต่ละพรรคชนะ (Top 10)

In [ ]:
winner_unit = partylist_scores.loc[
    partylist_scores.groupby("unit_key")["คะแนน"].idxmax() , ["unit_key", "พรรค"]
]

win_count = winner_unit["พรรค"].value_counts().head(TOP)
win_count_df = pd.DataFrame({"จำนวนหน่วย": win_count})
win_count_df["% จำนวนหน่วย"] = win_count / win_count.sum() * 100

display(win_count_df)

plt.figure(figsize=(10,6))
win_count.sort_values().plot(kind="barh")
plt.title(f"จำนวนหน่วยที่แต่ละพรรคชนะ (Top {TOP})")
plt.xlabel("จำนวนหน่วย")
plt.ylabel("พรรค")
plt.tight_layout()
plt.show()

### 3. สัดส่วนการใช้สิทธิเลือกตั้งสส.บัญชีรายชื่อรายตำบล

In [ ]:
turnout_by_tambon = (
    partylist_summary.groupby("ตำบล").apply(lambda x: pd.Series({
        "จำนวนหน่วยเลือกตั้ง": x["จำนวนบัตรทั้งหมด"].count(),
        "บัตรทั้งหมด": x["จำนวนบัตรทั้งหมด"].sum(),
        "บัตรที่ถูกใช้": (x["บัตรดี"] + x["บัตรเสีย"] + x["ไม่เลือกผู้ใด"]).sum(),
    }))
).reset_index()
turnout_by_tambon["% บัตรที่ถูกใช้"] = (turnout_by_tambon["บัตรที่ถูกใช้"] / turnout_by_tambon["บัตรทั้งหมด"] * 100).round(2)
turnout_by_tambon["บัตรที่เหลือ"] = turnout_by_tambon["บัตรทั้งหมด"] - turnout_by_tambon["บัตรที่ถูกใช้"]
turnout_by_tambon = turnout_by_tambon.sort_values("% บัตรที่ถูกใช้", ascending=False)

display(turnout_by_tambon)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(turnout_by_tambon["ตำบล"],
       turnout_by_tambon["บัตรที่ถูกใช้"],
       color="green",
       label="จำนวนบัตรที่ถูกใช้")
ax.bar(turnout_by_tambon["ตำบล"],
       turnout_by_tambon["บัตรที่เหลือ"],
       bottom=turnout_by_tambon["บัตรที่ถูกใช้"],
       color="red",
       label="จำนวนบัตรที่เหลือ")
ax.set_xlabel("ตำบล")
ax.set_ylabel("จำนวน (บัตร)")
ax.set_title("สัดส่วนการใช้สิทธิเลือกตั้งรายตำบล")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 4. การกระจายคะแนนพรรคในแต่ละตำบล (Top 10 ของแต่ละตำบล)

In [ ]:
top = partylist_scores.groupby("พรรค")["คะแนน"].sum().nlargest(TOP).index
df_top = partylist_scores[partylist_scores["พรรค"].isin(top)]

pivot = df_top.pivot_table(index="ตำบล", columns="พรรค", values="คะแนน", aggfunc="sum",fill_value=0)

pivot.plot(kind="bar", figsize=(14, 6))
plt.title(f"Top {TOP} พรรคในแต่ละตำบล")
plt.xticks(rotation=45)
plt.show()

In [ ]:
pivot = partylist_scores.pivot_table(index="ตำบล",columns="พรรค",values="คะแนน",aggfunc="sum",fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)

top = pivot_pct.sum().nlargest(TOP).index
pivot_pct = pivot_pct[top]

plt.figure(figsize=(12,6))
sns.heatmap(pivot_pct, cmap="Reds")
plt.title("Heatmap พรรค × ตำบล (%)")
plt.show()

### 5. บัตรเสียรายตำบล

In [ ]:
tambon_bad = (partylist_summary.groupby(["ตำบล"])
              .agg({"จำนวนบัตรทั้งหมด": "sum", "บัตรเสีย": "sum", "unit_key": "count"}).reset_index())

tambon_bad.rename(columns={"unit_key": "จำนวนหน่วย"}, inplace=True)

tambon_bad["% บัตรเสีย"] = (tambon_bad["บัตรเสีย"] / tambon_bad["จำนวนบัตรทั้งหมด"] * 100).replace([np.inf, -np.inf], np.nan)

total_bad = tambon_bad["บัตรเสีย"].sum()
tambon_bad = tambon_bad.sort_values("บัตรเสีย", ascending=False)
display(tambon_bad)

### 6. พรรคที่ชนะในแต่ละตำบล

In [ ]:
total_votes = partylist_scores.groupby('ตำบล')['คะแนน'].sum().reset_index(name='total_votes')
party_votes = partylist_scores.groupby(['ตำบล', 'พรรค'])['คะแนน'].sum().reset_index()

merged = party_votes.merge(total_votes, on='ตำบล')
merged['สัดส่วนคะแนน'] = merged['คะแนน'] / merged['total_votes']

top_party = merged.sort_values('สัดส่วนคะแนน', ascending=False).drop_duplicates('ตำบล')

party_colors = {p: c for p, c in zip(district_scores['พรรค'].unique(),plt.cm.tab10.colors)}

colors = top_party['พรรค'].map(party_colors)

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(top_party)), top_party['สัดส่วนคะแนน'], color=colors)

for i, v in enumerate(top_party['สัดส่วนคะแนน']):
    plt.text(i, v + 0.005, f"{v*100:.1f}%", ha='center', va='bottom', fontsize=8)

plt.xticks(range(len(top_party)), top_party['ตำบล'], rotation=45)
plt.title('พรรคที่ได้คะแนนสูงสุดในแต่ละตำบล')
plt.xlabel('ตำบล')
plt.ylabel('สัดส่วนคะแนน')

handles = [plt.Rectangle((0,0),1,1, color=c) for c in party_colors.values()]
plt.legend(handles, party_colors.keys(), title='พรรค')

plt.tight_layout()
plt.show()

### 7. 5 หน่วยฐานเสียงแข็ง และ 5 หน่วยที่ต้องพัฒนาของแต่ละพรรค (พรรคที่ได้คะแนน Top 10)

In [ ]:
TOP_UNIT = 5

top_parties = partylist_scores.groupby("พรรค")["คะแนน"].sum().nlargest(TOP).index

def calc_margin_partylist(group):
    group = group.sort_values("คะแนน", ascending=False).reset_index(drop=True)
    
    if len(group) < 2:
        return pd.Series({
            "winner": group.loc[0, "พรรค"],
            "winner_score": group.loc[0, "คะแนน"],
            "margin": group.loc[0, "คะแนน"],
            "margin_pct": 1.0
        })
    
    margin = group.loc[0, "คะแนน"] - group.loc[1, "คะแนน"]
    total = group["คะแนน"].sum()
    
    return pd.Series({
        "winner": group.loc[0, "พรรค"],
        "winner_score": group.loc[0, "คะแนน"],
        "margin": margin,
        "margin_pct": margin / total
    })

unit_margin = partylist_scores.groupby("unit_key").apply(calc_margin_partylist, include_groups=False).reset_index()
unit_margin = unit_margin.merge(partylist_scores[["unit_key", "หน่วย", "ตำบล", "อำเภอ"]].drop_duplicates(), on="unit_key")

for party in top_parties:
    own = unit_margin[unit_margin["winner"] == party].copy()
    lost = unit_margin[unit_margin["winner"] != party].copy()

    own_sorted = own.sort_values("margin_pct", ascending=False).head(TOP_UNIT).reset_index(drop=True)
    lost_sorted = lost.sort_values("margin_pct", ascending=False).head(TOP_UNIT).reset_index(drop=True)

    if own_sorted.empty and lost_sorted.empty:
        continue

    n_rows = max(len(own_sorted), len(lost_sorted), 1)
    fig_height = max(4, n_rows * 1.2)

    fig, axes = plt.subplots(1, 2, figsize=(12, fig_height))
    fig.suptitle(f"พรรค {party}", fontweight="bold")

    if not own_sorted.empty:
        own_sorted = own_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        print(own_sorted["margin_pct"].sort_values())
        values = own_sorted["margin_pct"] * 100
        axes[0].barh(own_sorted["ตำบล"] + " หน่วย " + own_sorted["หน่วย"], values, color="blue")
        axes[0].set_xlim(0, values.max() * 1.2)

    axes[0].set_xlabel("Margin (%)")
    axes[0].set_title(f"Top {TOP_UNIT} ฐานเสียงแข็ง")

    if not lost_sorted.empty:
        lost_sorted = lost_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        print(lost_sorted["margin_pct"].sort_values())
        values = lost_sorted["margin_pct"] * 100
        axes[1].barh(lost_sorted["ตำบล"] + " หน่วย " + lost_sorted["หน่วย"],values, color="red")
        axes[1].set_xlim(1, values.max() * 1.2)

    axes[1].set_xlabel("Margin (%)")
    axes[1].set_title(f"Top {TOP_UNIT} พื้นที่ที่แพ้ห่างที่สุด")

    plt.tight_layout()
    plt.show()

## 3. ประชามติ

### 1. เห็นชอบ/ไม่เห็นชอบ รายตำบล

In [ ]:
refer_tumbon = referendum_scores.groupby(["ตำบล", "รายการ"], as_index=False)["คะแนน"].sum()
r_pivot = refer_tumbon.pivot_table(index="ตำบล", columns="รายการ", values="คะแนน", fill_value=0)

if "เห็นชอบ" in r_pivot.columns:
    r_pivot["total_valid"] = r_pivot.sum(axis=1)
    r_pivot["pct_approve"] = (r_pivot["เห็นชอบ"] / r_pivot["total_valid"] * 100).round(2)

plot_cols = [c for c in ["เห็นชอบ", "ไม่เห็นชอบ", "ไม่แสดงความคิดเห็น"] if c in r_pivot.columns]
r_pivot["winner"] = r_pivot[plot_cols].idxmax(axis=1)
display(r_pivot)

plot_cols = [c for c in ["เห็นชอบ", "ไม่เห็นชอบ", "ไม่แสดงความคิดเห็น"] if c in r_pivot.columns]
r_pivot[plot_cols].plot(kind="bar", stacked=True, 
                        color=["green", "red", "grey"][:len(plot_cols)],
                        figsize=(12, 6))
plt.title("ผลประชามติรายตำบล")
plt.ylabel("คะแนน")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 2. สัดส่วนผู้มาใช้สิทธิประชามติรายตำบล

In [ ]:
ref = referendum_summary.groupby("ตำบล").agg({
    "ผู้มีสิทธิ": "sum", "ผู้มาใช้สิทธิ": "sum"}).reset_index()

ref["ผู้ไม่ใช้สิทธิ"] = ref["ผู้มีสิทธิ"] - ref["ผู้มาใช้สิทธิ"]
ref["% ใช้สิทธิ์"] = ref["ผู้มาใช้สิทธิ"] / ref["ผู้มีสิทธิ"]

display(ref)

x = np.arange(len(ref))
w = 0.6

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x, ref["ผู้มาใช้สิทธิ"], w, label="ผู้มาใช้สิทธิ", color="blue")
ax.bar(x, ref["ผู้ไม่ใช้สิทธิ"], w, bottom=ref["ผู้มาใช้สิทธิ"], label="ผู้ไม่ใช้สิทธิ", color="red")

ax.set_xticks(x)
ax.set_xticklabels(ref["ตำบล"], rotation=45, ha="right")
ax.set_ylabel("จำนวน (คน)")
ax.set_title("ผู้มาใช้สิทธิ์โหวตประชามติ")
ax.legend()

plt.tight_layout()
plt.show()